# InsPLAD training on Colab

Reproduces the two models behind [powerline-inspection-demo](https://github.com/Josh-E-S/powerline-inspection-demo). Before running: **Runtime → Change runtime type → GPU**. An L4 is fine for the 640 baseline and the classifier; the 1280 runs want an A100.

Checkpoints go to Google Drive, so a disconnected session loses nothing: rerun the setup cells, rerun the same training cell, and it resumes from `last.pt` automatically.

Every reported number came from these cells. Run 6c is the published detector.

In [ ]:
# 1. GPU check + persistent storage for checkpoints
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/powerline-runs'

## 2. Get the code

Clones the repository. The dataset is downloaded separately in the next step; it is not in the repo.

In [ ]:
REPO_URL = 'https://github.com/Josh-E-S/powerline-inspection-demo.git'

![ -d /content/powerline-inspection-demo ] || git clone $REPO_URL /content/powerline-inspection-demo
%cd /content/powerline-inspection-demo

## 3. Dataset

Pulled straight from the authors' Mendeley Data deposit (6.4 GB; datacenter bandwidth makes this a few minutes). Lands on Colab's fast local disk, which is also why prep's symlinks work: Drive mounts don't support them.

In [ ]:
!mkdir -p data/raw
!curl -L -C - -o /content/InsPLAD_Dataset.zip "https://data.mendeley.com/public-files/datasets/5n3fjgvfyz/files/96707044-99bb-40b2-bf23-6fa1b41ab9b0/file_downloaded"
!cd data/raw && unzip -q -o /content/InsPLAD_Dataset.zip InsPLAD-det.zip supervised_fault_classification.zip \
  && unzip -q -o InsPLAD-det.zip && unzip -q -o supervised_fault_classification.zip && rm *.zip
!ls data/raw

In [ ]:
# 4. Install deps (torch/torchvision are preinstalled on Colab) and stage the data
!pip install -q ultralytics onnx onnxruntime
!python3 scripts/prep_insplad.py

## 5. Smoke test first (~5 min total)

Tiny runs that exercise the entire pipeline: data loading, labels, training loop, checkpoints landing on Drive. If both finish and the `find` line lists `.pt` files under `powerline-runs/`, the full runs are safe to start. The point is to find path and permission mistakes in five minutes instead of thirty.

In [ ]:
!python3 scripts/train_detector.py --smoke --runs-dir $RUNS
!python3 scripts/train_classifier.py --smoke --runs-dir $RUNS
!find $RUNS -name '*.pt' | head

## 6. Detector runs

Three detector runs were done, in this order. Each is resumable, so rerun the same cell after a disconnect.

Run 6c is the one that produced the published results. If you only want to reproduce the headline numbers, run 6a to confirm the pipeline works, then skip to 6c.

In [ ]:
# 6a. Baseline: YOLO11-s @ 640, 120 epochs (~2.8 h on an L4)
# Test-set result: 0.734 Box AP / 0.893 AP50 (FP32)
!python3 scripts/train_detector.py --runs-dir $RUNS

In [ ]:
# 6b. Resolution + rare-class oversampling, default schedule (~2.6 h on an A100)
# Early stopping fired at epoch 67, so close_mosaic (epoch 111) never ran.
# Test-set result: 0.723 Box AP / 0.909 AP50 (FP32). Superseded by 6c.
# !python3 scripts/train_detector.py --imgsz 1280 --oversample --runs-dir $RUNS

In [ ]:
# 6c. FINAL MODEL. Same recipe as 6b but a 60-epoch schedule, so close_mosaic
# fires at epoch 50, and patience 0 so early stopping cannot skip it.
# ~2.3 h on an A100. Test-set result: 0.738 Box AP / 0.912 AP50 (FP32),
# 0.726 / 0.906 after INT8 quantization. This is the model in the README.
#
# Note: shortening the schedule also flips Ultralytics' optimizer=auto from
# MuSGD to AdamW. Pass --optimizer to pin it if you want to isolate that.
!python3 scripts/train_detector.py --imgsz 1280 --oversample --epochs 60 --patience 0 \
    --name det_yolo11s_1280_os_cm60 --runs-dir $RUNS

## 7. Condition classifier

EfficientNetV2-S on the 11 asset__condition classes; selects on val balanced accuracy. Fine on L4 (~30–45 min).

In [ ]:
!python3 scripts/train_classifier.py --runs-dir $RUNS

## 8. After training

Weights are on Drive under `powerline-runs/`:
- detector: `detect/<run-name>/weights/best.pt` (+ Ultralytics metrics/plots alongside)
- classifier: `classify/cls_effv2s/best.pt` (+ `metrics.jsonl`)

Download `best.pt` files locally for the export/quantization step (`scripts/export_quantize.py`). Record the GPU model shown by `nvidia-smi` for the README results table.

In [ ]:
# Training curves for a finished run. Swap RUN_NAME for the run you want:
#   det_yolo11s_640            (6a baseline)
#   det_yolo11s_1280_os        (6b, superseded)
#   det_yolo11s_1280_os_cm60   (6c, the published model)
RUN_NAME = 'det_yolo11s_1280_os_cm60'
from IPython.display import Image, display
display(Image(f'{RUNS}/detect/{RUN_NAME}/results.png'))